# 04 — Inference demo (загрузка своей песни)

**Что делает ноутбук:**
1. Загружает обученные best-чекпойнты CNN14 и ResNet18 (fold=0, seed=42).
2. Прогоняет inference на ваших MP3/WAV файлах.
3. Side-by-side вывод вероятностей для обеих моделей.
4. Запускает Gradio drag-and-drop UI с публичной ссылкой.

**Нужно:** прикрепить как Kaggle Dataset результаты ноутбуков 01 и 02 (best_*.pth), 
плюс свои аудио файлы (опционально).

In [ ]:
!pip install -q librosa==0.10.1 h5py soxr gradio 2>&1 | tail -3

In [ ]:
import sys
from pathlib import Path
for p in ['/kaggle/input/gtzan-cnn14-resnet18-src', '/kaggle/input/cnn14-resnet18-src',
          '/kaggle/working', str(Path.cwd().parent)]:
    if Path(p, 'src', '__init__.py').exists():
        sys.path.insert(0, p)
        break

import torch
from src.predict import load_model, predict_audio, launch_gradio
from src import GENRES
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device={device}')

In [ ]:
# Поиск чекпойнтов
def find_ckpt(name: str) -> Path | None:
    for d in [Path('/kaggle/input/gtzan-cnn14-results/outputs'),
              Path('/kaggle/input/gtzan-resnet18-results/outputs'),
              Path('/kaggle/working/outputs'),
              Path.cwd().parent / 'outputs']:
        if (d / name).exists():
            return d / name
    return None

CK_CNN14 = find_ckpt('best_cnn14_fold0_seed42.pth')
CK_RESNET18 = find_ckpt('best_resnet18_fold0_seed42.pth')
assert CK_CNN14 and CK_RESNET18, 'Чекпойнты не найдены. Сначала запустите 01 и 02.'
print(f'CNN14: {CK_CNN14}')
print(f'ResNet18: {CK_RESNET18}')

In [ ]:
# Загружаем модели
m_cnn14 = load_model('cnn14', CK_CNN14, device=device)
m_resnet18 = load_model('resnet18', CK_RESNET18, channel_strategy='avg_conv1', device=device)
print('Модели загружены, готовы к инференсу')

In [ ]:
# Программный inference на одном файле (примените на любых mp3/wav)
# Подключите свои аудио как Kaggle Dataset, либо upload через File Browser.
test_audio_candidates = [
    '/kaggle/input/my-audio/song.mp3',
    '/kaggle/working/test.mp3',
    str(Path.cwd().parent / 'GTZAN' / 'genres_original' / 'rock' / 'rock.00000.wav'),
]
audio_path = next((p for p in test_audio_candidates if Path(p).exists()), None)
if audio_path:
    print(f'Анализ: {audio_path}')
    r_c = predict_audio(m_cnn14, audio_path, device=device)
    r_r = predict_audio(m_resnet18, audio_path, device=device)
    print('\n--- CNN14 (предобучение AudioSet) ---')
    print(f'Жанр: {r_c["top_genre"]} ({r_c["top_probability"]*100:.1f}%)')
    for k in r_c['top_k']:
        print(f'  {k["genre"]:10s} {k["probability"]*100:5.1f}%')
    print('\n--- ResNet18 (предобучение ImageNet) ---')
    print(f'Жанр: {r_r["top_genre"]} ({r_r["top_probability"]*100:.1f}%)')
    for k in r_r['top_k']:
        print(f'  {k["genre"]:10s} {k["probability"]*100:5.1f}%')
else:
    print('Не найдено тестовое аудио. Загрузите файл и обновите test_audio_candidates.')

In [ ]:
# Визуализация: bar chart вероятностей по 10 жанрам
if audio_path:
    import matplotlib.pyplot as plt
    import numpy as np
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(GENRES))
    width = 0.4
    p_c = [r_c['all_probs'][g] for g in GENRES]
    p_r = [r_r['all_probs'][g] for g in GENRES]
    ax.bar(x - width/2, p_c, width, label='CNN14', color='#1f77b4')
    ax.bar(x + width/2, p_r, width, label='ResNet18', color='#ff7f0e')
    ax.set_xticks(x)
    ax.set_xticklabels(GENRES, rotation=30)
    ax.set_ylabel('Probability')
    ax.set_title(f'Distribution per genre for: {Path(audio_path).name}')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# === Gradio UI с публичной ссылкой ===
# Для запуска в Kaggle Notebook нужно включить Settings → Internet ON.
# Gradio создаст публичный URL вида https://xxxxxxxx.gradio.live
launch_gradio(
    weights_cnn14=CK_CNN14,
    weights_resnet18=CK_RESNET18,
    channel_strategy='avg_conv1',
    share=True,
)

## Использование на защите

1. Запустите эту ячейку — появится публичная ссылка `*.gradio.live` (живёт 72 часа).
2. Откройте её в браузере / покажите на проекторе.
3. Перетащите песню → обе модели вернут свои top-3 жанров side-by-side.
4. Покажите случаи, где модели расходятся — это материал для дискуссии.